In [1]:
import os, sys, json
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT  = r'C:\Users\chiku\OneDrive\Documents\ecom_dynamic_pricing_optimization'
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')

# Load data and column definitions
train = pd.read_parquet(os.path.join(PROCESSED_DIR, 'train.parquet'))
with open(os.path.join(PROCESSED_DIR, 'col_definitions.json')) as f:
    cols = json.load(f)

Y_col = cols['Y_col']
T_col = cols['T_col']
W_cols = cols['W_cols']
X_cols = cols['X_cols']

print(f'Loaded train: {train.shape}')
print(f'Y={Y_col}, T={T_col}, W={len(W_cols)} controls, X={len(X_cols)} het. features')

Loaded train: (500000, 20)
Y=log_units, T=log_price, W=9 controls, X=3 het. features


In [2]:
import statsmodels.api as sm
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_absolute_percentage_error, r2_score
from sklearn.model_selection import train_test_split

# ─── Train/test split ─────────────────────────────────────────────────────────
X_features = [T_col] + W_cols
X = train[X_features].fillna(0)
y = train[Y_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

# ─── Model 1: Naive OLS (price only — no controls) ────────────────────────────
X_naive = sm.add_constant(train[T_col])
ols_naive = sm.OLS(y, X_naive).fit()
naive_elasticity = ols_naive.params[T_col]

# ─── Model 2: OLS with controls ───────────────────────────────────────────────
X_full = sm.add_constant(train[X_features].fillna(0))
ols_full = sm.OLS(y, X_full).fit()
ols_elasticity = ols_full.params[T_col]

# ─── Model 3: Ridge regression ────────────────────────────────────────────────
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
ridge_elasticity = ridge.coef_[0]  # First coef = log_price
ridge_mape = mean_absolute_percentage_error(y_test, ridge.predict(X_test))
ridge_r2   = r2_score(y_test, ridge.predict(X_test))

# ─── Results ──────────────────────────────────────────────────────────────────
print('\n' + '=' * 55)
print('BASELINE MODEL RESULTS')
print('=' * 55)
print(f'{"Model":<30} {"Elasticity":>12} {"Notes"}')
print('-' * 55)
print(f'{"Naive OLS (no controls)":<30} {naive_elasticity:>12.4f} {"← most biased"}')
print(f'{"OLS + 9 controls":<30} {ols_elasticity:>12.4f} {"← better but still biased"}')
print(f'{"Ridge + 9 controls":<30} {ridge_elasticity:>12.4f} {"← regularised"}')
print('-' * 55)
print(f'\nRidge test MAPE:  {ridge_mape:.4f}')
print(f'Ridge test R²:    {ridge_r2:.4f}')
print(f'\n⚠️  All these estimates are biased — DML corrects this in notebook 04')
print(f'   Bias = difference between naive OLS and DML (computed in notebook 04)')

Train: (400000, 10), Test: (100000, 10)

BASELINE MODEL RESULTS
Model                            Elasticity Notes
-------------------------------------------------------
Naive OLS (no controls)              0.0045 ← most biased
OLS + 9 controls                     0.0058 ← better but still biased
Ridge + 9 controls                   0.0054 ← regularised
-------------------------------------------------------

Ridge test MAPE:  0.4292
Ridge test R²:    0.0176

⚠️  All these estimates are biased — DML corrects this in notebook 04
   Bias = difference between naive OLS and DML (computed in notebook 04)


In [ ]:
# Save baseline results for comparison in notebook 04
import json
baseline_results = {
    'naive_ols_elasticity': round(naive_elasticity, 6),
    'ols_controls_elasticity': round(ols_elasticity, 6),
    'ridge_elasticity': round(ridge_elasticity, 6),
    'ridge_mape': round(ridge_mape, 6),
    'ridge_r2': round(ridge_r2, 6),
}
with open(os.path.join(PROCESSED_DIR, 'baseline_results.json'), 'w') as f:
    json.dump(baseline_results, f, indent=2)

print('Baseline results saved ✅')
print('\nNotebook 03 complete — ready for notebook 04 (Double ML)')

Baseline results saved ✅

Notebook 03 complete — ready for notebook 04 (Double ML)


: 